# PD 分离技术概述

从本节开始，我们将深入学习 PD 分离架构的原理、优势以及面临的 KV Cache 传输挑战，为后续学习 llm-datadist 接口奠定理论基础。

本节学习大纲如下：

- 传统融合部署的瓶颈
- PD 分离架构的原理
- PD 分离的核心优势
- PD 分离面临的 KV Cache 传输挑战
- HIXL 在 PD 分离中的角色


## 1. 传统融合部署的瓶颈

### 1.1 Prefill 与 Decode 的计算特征差异

大模型推理过程分为 Prefill（预填充）和 Decode（解码）两个阶段，两者的计算特征存在显著差异：

| 维度 | Prefill 阶段 | Decode 阶段 |
| --- | --- | --- |
| 计算量 | 高（处理整个 prompt） | 低（逐 token 生成） |
| 显存占用 | KV Cache 逐层增长 | KV Cache 已固定 |
| 计算密度 | 计算密集型（compute-bound） | 访存密集型（memory-bound） |
| 算术强度 | 高（FLOPs/字节 比值大） | 低（FLOPs/字节 比值小） |
| 批处理能力 | 难以大批量（显存限制） | 适合大批量（显存空闲） |
| 延迟特征 | TTFT（首 token 延迟） | TPOT（每 token 延迟） |
| 优化方向 | 提升算力利用率 | 提升显存带宽利用率 |

从 Roofline 模型角度分析：

- **Prefill 阶段**：处理长度为 $S$ 的 prompt，计算量约为 $2 \times S^2 \times \text{params}$，而访存量约为 $S \times \text{params}$，算术强度约 $2S$。当 $S$ 较大时（如 2048），算术强度远超 NPU 算力/带宽比，属于计算密集型，瓶颈在算力。
- **Decode 阶段**：每步生成 1 个 token，计算量约为 $2 \times S \times \text{params}$，访存量约为 $S \times \text{params}$，算术强度约 2。远低于 NPU 算力/带宽比，属于访存密集型，瓶颈在显存带宽。

这意味着两个阶段对硬件资源的需求方向完全不同：Prefill 需要高算力，Decode 需要高带宽，融合部署无法同时满足。

下图直观对比了两个阶段的计算特征差异，这正是 PD 分离架构的核心动机：

<img src="./images/prefill_decode_characteristics.svg" alt="Prefill 与 Decode 计算特征对比图" width="90%">

### 1.2 融合部署的问题

在传统融合部署中，Prefill 和 Decode 运行在同一组 NPU 节点上，带来以下问题：

- Prefill 和 Decode 共享同一批 GPU/NPU 资源，互相抢占
- Prefill 大批量计算时导致 Decode 请求排队，TPOT 抖动
- Decode 占用显存时限制 Prefill 批量大小，降低吞吐
- 资源配置无法针对不同阶段做独立优化

**干扰示例**：假设某融合节点同时处理 1 个 Prefill 请求（batch_size=32，seq_len=2048）和若干 Decode 请求。Prefill 阶段大量矩阵乘计算占用算力单元，导致 Decode 的逐 token 生成被阻塞，TPOT 从期望的 50ms 飙升至 300ms+，用户体验明显下降。同时，Prefill 的 32 batch KV Cache 占用大量显存，迫使 Decode 请求的 batch size 从 128 降至 32，吞吐量损失 75%。

### 1.3 行业趋势：分离式推理服务

PD 分离并非华为独有概念，业界主流推理框架均已跟进：

| 框架/项目 | PD 分离支持 | 特点 |
| --- | --- | --- |
| vLLM (Disaggregated Prefill) | 支持分离式 prefill | 学术论文驱动，工业界标杆 |
| SGLang (Disaggregated Serving) | 支持 P/D 分离 | 基于 RDMA 的 KV Cache 传输 |
| Mooncake (Moonshot AI) | KVCache-centric 分离架构 | 分离池化 KV Cache 存储 |
| DeepLink | 分离式推理 | 昇腾生态兼容 |
| HIXL + omni-infer | xPyD 交叉分离 | 昇腾 NPU 原生，HCCS 高带宽 |

HIXL 的优势在于原生支持昇腾硬件的 HCCS 高带宽链路（最高 119GB/s），相比通用 RDMA 方案带宽提升数倍，且提供 KV Cache 语义级的高层接口，降低开发复杂度（HIXL 的详细架构将在第 5 节展开）。

> **小结**：Prefill 与 Decode 计算特征的根本差异，使得融合部署面临资源竞争困境。业界主流方案是将其分离部署——下一节将详细讲解 PD 分离架构的原理与工作流程。


## 2. PD 分离架构的原理

### 2.1 架构示意

PD 分离架构将 Prefill 和 Decode 部署到不同的节点集群，Prefill 节点专注处理 prompt 生成 KV Cache，完成后将 KV Cache 传输到 Decode 节点，Decode 节点利用接收到的 KV Cache 继续生成后续 token。

<img src="./images/pd_separation_architecture.svg" alt="PD 分离架构图" width="90%">

### 2.2 工作流程

PD 分离的完整工作流程如下：

- 步骤 1：请求路由到 Prefill 节点
- 步骤 2：Prefill 节点执行前向计算，生成 KV Cache
- 步骤 3：将 KV Cache 从 Prefill 节点传输到 Decode 节点
- 步骤 4：Decode 节点接收 KV Cache，继续逐 token 解码
- 步骤 5：Decode 节点返回生成结果

### 2.3 xPyD（交叉 Prefill-Decode）

xPyD（交叉 Prefill-Decode）是 PD 分离的进阶形态，支持 M 个 Prefill 节点对应 N 个 Decode 节点的交叉调度（M:N 拓扑）。相比简单的 1P1D 固定配对，xPyD 具有以下优势：

- **灵活的 P/D 比例**：可根据负载动态调整 Prefill 和 Decode 节点数量（如 2P1D、4P1D、8P1D）
- **负载分担**：一个 Decode 节点可从多个 Prefill 节点拉取不同请求的 KV Cache
- **容错性**：某个 Prefill 节点故障时，请求可路由到其他 Prefill 节点

在 llm-datadist 中，xPyD 通过**单向链接 + pull_blocks**模式实现：

- 每个 Prefill 节点设置监听 IP（`listen_ip_info`），作为 Server 等待 Decode 连接
- 每个 Decode 节点通过 `link_clusters` 向所有需要通信的 Prefill 节点发起单向链接
- Decode 端通过 `pull_blocks` 按需从指定 Prefill 节点拉取 KV Cache 分块

下图展示了 xPyD 的 M:P → N:D 交叉拓扑，每个 Decode 节点通过单向链接与所有 Prefill 节点建立通信，按需拉取不同请求的 KV Cache 分块：

<img src="./images/xpyd_topology.svg" alt="xPyD 交叉分离拓扑图" width="90%">

### 2.4 典型部署拓扑

omni-infer 支持多种 PD 分离部署规格，适配不同规模场景：

| 部署规格 | 机器数 | P:D 比例 | 适用场景 |
| --- | --- | --- | --- |
| 1P1D | 3 台（含 1 台 Proxy） | 1:1 | 开发测试、小规模验证 |
| 2P1D | 4 台 | 2:1 | Prefill 密集场景（长 prompt） |
| 4P1D | 8 台 | 4:1 | 中等规模生产 |
| 8P1D | 16 台 | 8:1 | 大规模生产，高吞吐 |
| EP144 18P1D | 36 台 | 18:1 | MoE 超大规模（专家并行 144 卡） |

选择 P:D 比例时需考虑：
- Prefill 计算量大，通常需要更多算力，P 节点比例较高
- Decode 可通过增大 batch size 提升吞吐，D 节点可较少
- MoE 模型专家并行需要大量 NPU，EP144 需要 18 台 8 卡机器


## 3. PD 分离的核心优势

PD 分离架构通过资源隔离和独立优化，带来以下核心优势：

| 优势 | 说明 |
| --- | --- |
| 资源独立优化 | Prefill 节点可配置高算力，Decode 节点可配置高显存 |
| 批处理提升 | Prefill 和 Decode 各自独立批处理，互不干扰 |
| 弹性扩缩容 | P 和 D 节点可独立扩缩容，按需调整比例 |
| 显存利用率 | Decode 节点可利用空闲显存增大 batch size |
| TTFT/TPOT 解耦 | 分别优化首 token 延迟和每 token 延迟 |
| 硬件异构利用 | Prefill 可用高算力芯片，Decode 可用高带宽芯片 |
| 故障隔离 | Prefill 和 Decode 故障互不影响，提升服务可用性 |

然而，分离部署也引入了新的技术难题——Prefill 节点生成的 KV Cache 必须高效传输到 Decode 节点，这对带宽、延迟和内存管理提出了严峻挑战。下一节将逐一剖析这些挑战。


## 4. PD 分离面临的 KV Cache 传输挑战

### 4.1 传输数据量大

KV Cache 的大小与模型规模、batch size 和序列长度直接相关：

```
KV Cache 大小 = num_layers × 2 (K+V) × batch_size × seq_len × num_heads × head_dim × dtype_size
```

以 **LLaMA-65B**（80 层，64 头，head_dim=128，FP16）为例，计算单请求 KV Cache 大小：

```
# 单层单个请求（K+V）
单层 = 2 × 1 (batch) × 2048 (seq_len) × 64 (heads) × 128 (head_dim) × 2 (FP16)
     = 2 × 2048 × 64 × 128 × 2 = 67,108,864 字节 = 64 MB

# 80 层
单请求总量 = 64 MB × 80 = 5120 MB ≈ 5 GB

# batch_size=8
总传输量 = 5 GB × 8 = 40 GB
```

40 GB 的数据传输对带宽提出极高要求：

- 以 10Gbps 以太网传输需要约 32 秒（远超可接受延迟）
- 以 100Gbps RoCE（实际约 12.5GB/s）传输需要约 3.2 秒
- 以昇腾 HCCS（119GB/s）传输仅需约 0.34 秒

### 4.2 传输延迟敏感

KV Cache 传输时间直接影响 TTFT（首 token 延迟），需要在 Prefill 计算完成后尽快完成传输。理想情况下，传输应与 Prefill 计算重叠以隐藏传输延迟。

**逐层异步传输**是关键优化：Prefill 每算完一层的 KV Cache 立即启动该层传输，不等全部层算完再传。这样传输时间被计算时间掩盖，整体 TTFT 接近无传输的理想值。

下图对比了传统“全部算完再传”与逐层异步传输的差异，后者通过计算与传输重叠显著降低端到端延迟：

<img src="./images/layer_async_transfer.svg" alt="逐层异步传输时序图" width="90%">

### 4.3 内存管理复杂

- Paged Attention 场景下 KV Cache 非连续存储，需要支持分块（blocks）传输
- 需要支持逐层（per-layer）异步传输，实现计算与传输重叠
- 多节点场景下的 Cache 合并与管理
- Device 内存与 Host 内存的灵活放置（CachePlacement）
- 内存地址对齐要求（HCCS 要求 2MB 对齐）

### 4.4 多节点调度

xPyD 场景下一个 Decode 节点可能需要从多个 Prefill 节点拉取 KV Cache，涉及复杂的路由和调度逻辑。需要解决：

- 请求到 Prefill 节点的路由策略（最小化 TTFT）
- Decode 节点的负载均衡（平衡活跃请求数和 token 数）
- P+D 联合评分优化（共享内存联合调度）

### 4.5 带宽需求分析

不同传输链路的带宽差异显著，直接影响 PD 分离的可行性和性能：

| 链路类型 | 带宽（实测） | 适用场景 |
| --- | --- | --- |
| 以太网（10Gbps） | ~1.25 GB/s | 不可行，延迟过高 |
| RoCE/RDMA（100Gbps） | ~12.5 GB/s | 跨节点，延迟可接受 |
| 昇腾 HCCS D2D | 最高 119 GB/s | 机内 D2D，最优性能 |
| 昇腾 RDMA（A3） | ~22 GB/s（128MB 数据） | 跨机，性能良好 |

HIXL 在 A3 硬件上实测：128MB 数据块 HCCS 带宽达 119GB/s，RDMA 带宽达 22GB/s。这意味着同样 40GB KV Cache 传输，HCCS 仅需 0.34s，而 RDMA 需 1.8s。

下图以 40GB KV Cache 传输为例，直观对比了不同链路的带宽差异与传输耗时：

<img src="./images/bandwidth_comparison.svg" alt="PD 分离传输链路带宽对比图" width="90%">

### 4.6 异构设备传输

昇腾硬件代际之间存在差异，PD 分离需要跨代际兼容：

| 硬件型号 | 传输方式 | 特点 |
| --- | --- | --- |
| Atlas 800I A2 / A200I A2 Box | HCCS D2D | 仅支持机内 D2D |
| Atlas A3 | HCCS | 不支持 Host 内存作为远端 Cache |
| Atlas A5 / 950PR / 950DT | UB / RoCE | 超节点内 UB，跨超节点 RoCE |

HIXL 屏蔽底层硬件差异，上层 llm-datadist 接口保持一致，实现 A2/A3/A5 跨代际无缝迁移。

> **小结**：PD 分离虽优势显著，但 KV Cache 传输面临数据量大、延迟敏感、内存管理复杂、多节点调度、带宽瓶颈和异构兼容六大挑战。下一节将介绍 HIXL 如何通过单边通信和高层语义接口系统性地解决这些问题。


## 5. HIXL 在 PD 分离中的角色

### 5.1 HIXL 概述

HIXL（Huawei Xfer Library）是华为面向集群的单边通信库，专为 PD 分离场景的 KV Cache 传输设计。HIXL 具有三大核心优势：

| 优势 | 说明 |
| --- | --- |
| 单边零拷贝（One-Sided Zero-Copy） | 无需远端节点参与，直接读写远端内存，实现计算与传输重叠，避免冗余数据搬运 |
| 屏蔽硬件差异兼容多链路 | 支持 HCCS/RDMA/FabricMem，带宽最高 119GB/s，A2/A3/A5 跨代际无缝兼容 |
| 极简 API | 10+ 核心调用，提供 C++/Python 接口，已集成 Mooncake/DeepLink/vLLM/SGLang 生态 |

### 5.2 两大组件架构

HIXL 采用分层架构，包含两大核心组件：

| 组件 | 职责 | 接口 |
| --- | --- | --- |
| HIXL Engine | 底层基础传输引擎，提供 D2D/D2H/H2D 通信原语，支持 HCCS/RDMA 链路 | C++/Python 基础传输 API |
| LLM-DataDist | 高层 KV Cache 语义库，封装集群管理、内存注册、Pull/Push 传输 | C++/Python KV Cache API |

LLM-DataDist 构建在 HIXL Engine 之上，提供面向 PD 分离场景的高层语义接口（如 `PullKvCache`、`PushKvBlocks`、`transfer_cache_async`），开发者无需关心底层传输细节。

### 5.3 CS 通信模型

HIXL 底层基于 Client-Server（CS）单边通信模型：

- **Server（服务端）**：创建 endpoint，注册并导出本地内存，监听连接
- **Client（客户端）**：连接到 Server，匹配 endpoint，导入远端内存，执行 Put/Get

在 PD 分离场景中的映射：

| llm-datadist 语义 | HIXL CS 操作 | 角色 | 数据流向 |
| --- | --- | --- | --- |
| PullKvCache / PullKvBlocks | Get | Decode(Client) 读 Prefill(Server) | Prefill → Decode |
| PushKvCache / PushKvBlocks | Put | Decode(Client) 写 Prefill(Server) 或反向 | 按需 |

关键点：单边通信意味着 Put/Get 操作由 Client 发起，**Server 无需参与数据搬运**，Server 的 CPU 可以专注 Prefill 计算，实现计算与传输重叠。

### 5.4 生态集成

HIXL 已与主流推理框架和传输库集成：

| 集成方 | 类型 | 说明 |
| --- | --- | --- |
| Mooncake (Moonshot AI) | KVCache 池化 | HIXL 作为传输后端 |
| DeepLink | 推理框架 | 昇腾生态分离式推理 |
| vLLM | 主流推理框架 | Disaggregated Prefill 支持 |
| SGLang | 高性能推理框架 | Disaggregated Serving 支持 |
| omni-infer | 昇腾推理加速包 | 原生集成 llm-datadist |

据实测，HIXL 可降低内存访问延迟约 20%，显著提升 PD 分离场景的端到端推理性能。

后续章节将详细讲解 llm-datadist 的接口体系和使用方法。


## 课后练习

本节介绍了 PD 分离架构的动机、原理、优势以及 KV Cache 传输挑战，请根据学习内容完成以下题目进行自测。

1. （判断题）PD 分离架构中，Prefill 节点和 Decode 节点共享同一批 NPU 资源。

2. （判断题）Prefill 阶段是计算密集型，Decode 阶段是访存密集型。

3. （单选题）PD 分离架构中，KV Cache 传输发生在哪个阶段之间？
    A. 请求路由之前
    B. Prefill 计算完成后、Decode 计算开始前
    C. Decode 计算完成后
    D. 贯穿整个推理过程

4. （单选题）以下哪项不是 PD 分离架构的核心优势？
    A. 资源独立优化
    B. 批处理提升
    C. 降低模型参数量
    D. 弹性扩缩容

5. （多选题）PD 分离场景下 KV Cache 传输面临哪些挑战？
    A. 传输数据量大
    B. 传输延迟敏感
    C. 内存管理复杂（分块、逐层传输）
    D. 多节点调度复杂

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/05.02_answer.txt